In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

In [3]:
#Paths
UPAG_DIR = Path("../data/raw/upag")

FILES = {
    "Rice": UPAG_DIR / "rice_upag_raw.csv",
    "Wheat": UPAG_DIR / "wheat_upag_raw.csv",
    "Maize": UPAG_DIR / "maize_upag_raw.csv",
    "Urad": UPAG_DIR / "urad_upag_raw.csv",
}

#Inspecting every file
for crop, path in FILES.items():

    df = pd.read_csv(path)

    print("=" * 70)
    print(crop)
    print("=" * 70)

    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

Rice
Shape: (782, 18)

Columns:
['objectid', 'uid', 'district', 'state', 'lgd_distcode', 'lgd_statecode', 'cencode2011', 'ricearea25', 'ricearea24', 'ricearea23', 'riceprod25', 'riceprod24', 'riceprod23', 'riceyld25', 'riceyld24', 'riceyld23', 'st_area(shape)', 'st_perimeter(shape)']
Wheat
Shape: (782, 18)

Columns:
['objectid', 'uid', 'district', 'state', 'lgd_distcode', 'lgd_statecode', 'cencode2011', 'wheatarea25', 'wheatarea24', 'wheatarea23', 'wheatprod25', 'wheatprod24', 'wheatprod23', 'wheatyld25', 'wheatyld24', 'wheatyld23', 'st_area(shape)', 'st_perimeter(shape)']
Maize
Shape: (782, 18)

Columns:
['objectid', 'uid', 'district', 'state', 'lgd_distcode', 'lgd_statecode', 'cencode2011', 'mazarea25', 'mazarea24', 'mazarea23', 'mazprod25', 'mazprod24', 'mazprod23', 'mazyld25', 'mazyld24', 'mazyld23', 'st_area(shape)', 'st_perimeter(shape)']
Urad
Shape: (782, 18)

Columns:
['objectid', 'uid', 'district', 'state', 'lgd_distcode', 'lgd_statecode', 'cencode2011', 'uradarea25', 'uradare

In [4]:
#Verifying geographic identifiers
for crop, path in FILES.items():

    df = pd.read_csv(path)

    print(
        crop,
        "districts =", df["lgd_distcode"].nunique(),
        "states =", df["lgd_statecode"].nunique(),
        "missing district codes =",
        df["lgd_distcode"].isna().sum(),
        "missing state codes =",
        df["lgd_statecode"].isna().sum()
    )

Rice districts = 782 states = 36 missing district codes = 0 missing state codes = 0
Wheat districts = 782 states = 36 missing district codes = 0 missing state codes = 0
Maize districts = 782 states = 36 missing district codes = 0 missing state codes = 0
Urad districts = 782 states = 36 missing district codes = 0 missing state codes = 0


In [ ]:
#Inspecting data
upag = pd.read_csv(
    "../data/processed/crop_yield/"
    "upag_2022_23_2024_25.csv"
)

print(upag.shape)

print(
    upag.groupby(
        ["crop", "year"]
    ).size()
)

(9384, 16)
crop   year     
Maize  2022-2023    782
       2023-2024    782
       2024-2025    782
Rice   2022-2023    782
       2023-2024    782
       2024-2025    782
Urad   2022-2023    782
       2023-2024    782
       2024-2025    782
Wheat  2022-2023    782
       2023-2024    782
       2024-2025    782
dtype: int64


In [6]:
#Validating the yeild
upag["calculated_yield_kg_ha"] = (
    upag["production_tonnes"]
    / upag["area_ha"]
    * 1000
)

upag["yield_difference"] = (
    upag["yield_kg_ha"]
    - upag["calculated_yield_kg_ha"]
)

valid = (
    upag["area_ha"].notna()
    & (upag["area_ha"] > 0)
    & upag["production_tonnes"].notna()
    & upag["yield_kg_ha"].notna()
)

print(
    upag.loc[
        valid,
        "yield_difference"
    ].describe()
)

count    5575.000000
mean       27.130511
std       325.754488
min     -3811.000000
25%       -20.000000
50%         0.882353
75%        29.000000
max      5744.000000
Name: yield_difference, dtype: float64


In [7]:
#Inspecting DES seasons
import pandas as pd

des = pd.read_csv(
    "../data/processed/crop_yield/"
    "des_apy_2013_14_2022_23.csv"
)

des_2022 = des[
    des["year"] == "2022-2023"
].copy()

print(
    des_2022.groupby(
        ["crop", "season"]
    ).size()
)

crop       season    
Arhar/Tur  Kharif        495
           Rabi           82
           Total          59
           Whole Year      1
Bajra      Kharif        347
                        ... 
Wheat      Kharif          5
           Rabi          537
           Summer          7
           Total           5
           Whole Year      9
Length: 64, dtype: int64


In [8]:
print(
    des_2022[
        des_2022["crop"].isin(
            ["Rice", "Wheat", "Maize", "Urad"]
        )
    ]["season"]
    .value_counts()
)

season
Kharif        1530
Rabi          1242
Total         1099
Summer         628
Autumn         265
Winter         231
Whole Year      14
Name: count, dtype: int64


In [9]:
for crop in ["Rice", "Wheat", "Maize", "Urad"]:

    print("\n", "=" * 50)
    print(crop)

    print(
        des_2022[
            des_2022["crop"] == crop
        ]["season"]
        .value_counts()
    )


Rice
season
Kharif    485
Total     363
Summer    246
Winter    193
Autumn    176
Rabi      115
Name: count, dtype: int64

Wheat
season
Rabi          537
Whole Year      9
Summer          7
Kharif          5
Total           5
Name: count, dtype: int64

Maize
season
Kharif        552
Total         393
Rabi          351
Summer        214
Autumn         74
Winter         10
Whole Year      5
Name: count, dtype: int64

Urad
season
Kharif    488
Total     338
Rabi      239
Summer    161
Winter     28
Autumn     15
Name: count, dtype: int64


In [10]:
#Inspecting UPAG data
upag = pd.read_csv(
    "../data/processed/crop_yield/"
    "upag_2022_23_2024_25.csv"
)

upag_2022 = upag[
    upag["year"] == "2022-2023"
].copy()

print(
    upag_2022.groupby("crop").size()
)

crop
Maize    782
Rice     782
Urad     782
Wheat    782
dtype: int64


In [11]:
#Checking geographic codes
print(
    des_2022[
        [
            "state_code",
            "district_code"
        ]
    ].dtypes
)

print(
    upag_2022[
        [
            "state_code",
            "district_code"
        ]
    ].dtypes
)

state_code       int64
district_code    int64
dtype: object
state_code       int64
district_code    int64
dtype: object


In [12]:
print(
    "DES states:",
    des_2022["state_code"].nunique()
)

print(
    "UPAg states:",
    upag_2022["state_code"].nunique()
)

print(
    "DES districts:",
    des_2022["district_code"].nunique()
)

print(
    "UPAg districts:",
    upag_2022["district_code"].nunique()
)

DES states: 34
UPAg states: 36
DES districts: 729
UPAg districts: 782


In [13]:
#DES-UPAG geographic overlap
import pandas as pd

des = pd.read_csv(
    "../data/processed/crop_yield/"
    "des_apy_2013_14_2022_23.csv"
)

upag = pd.read_csv(
    "../data/processed/crop_yield/"
    "upag_2022_23_2024_25.csv"
)

des_2022 = des[
    des["year"] == "2022-2023"
].copy()

upag_2022 = upag[
    upag["year"] == "2022-2023"
].copy()

print("DES 2022-23:", des_2022.shape)
print("UPAg 2022-23:", upag_2022.shape)

DES 2022-23: (9742, 16)
UPAg 2022-23: (3128, 16)


In [14]:
#checking state code overlap
des_states = set(
    des_2022["state_code"].dropna().astype(int)
)

upag_states = set(
    upag_2022["state_code"].dropna().astype(int)
)

common_states = (
    des_states & upag_states
)

des_only_states = (
    des_states - upag_states
)

upag_only_states = (
    upag_states - des_states
)

print("DES states:", len(des_states))
print("UPAg states:", len(upag_states))
print("Common states:", len(common_states))

print("\nDES-only state codes:")
print(sorted(des_only_states))

print("\nUPAg-only state codes:")
print(sorted(upag_only_states))

DES states: 34
UPAg states: 36
Common states: 34

DES-only state codes:
[]

UPAg-only state codes:
[7, 31]


In [15]:
#checking district code overlap
des_districts = set(
    des_2022["district_code"]
    .dropna()
    .astype(int)
)

upag_districts = set(
    upag_2022["district_code"]
    .dropna()
    .astype(int)
)

common_districts = (
    des_districts & upag_districts
)

des_only_districts = (
    des_districts - upag_districts
)

upag_only_districts = (
    upag_districts - des_districts
)

print("DES districts:", len(des_districts))
print("UPAg districts:", len(upag_districts))
print("Common districts:", len(common_districts))

print(
    "DES-only districts:",
    len(des_only_districts)
)

print(
    "UPAg-only districts:",
    len(upag_only_districts)
)

DES districts: 729
UPAg districts: 782
Common districts: 729
DES-only districts: 0
UPAg-only districts: 53


In [16]:
#calculating actual geographic coverage
des_coverage = (
    len(common_districts)
    / len(des_districts)
    * 100
)

upag_coverage = (
    len(common_districts)
    / len(upag_districts)
    * 100
)

print(
    f"DES district coverage: "
    f"{des_coverage:.2f}%"
)

print(
    f"UPAg district coverage: "
    f"{upag_coverage:.2f}%"
)

DES district coverage: 100.00%
UPAg district coverage: 93.22%


In [17]:
#Crop level geographic overlap
crops = [
    "Rice",
    "Wheat",
    "Maize",
    "Urad"
]

for crop in crops:

    des_crop = des_2022[
        (des_2022["crop"] == crop)
        &
        (des_2022["season"] == "Total")
    ]

    upag_crop = upag_2022[
        upag_2022["crop"] == crop
    ]

    des_codes = set(
        des_crop["district_code"]
        .dropna()
        .astype(int)
    )

    upag_codes = set(
        upag_crop["district_code"]
        .dropna()
        .astype(int)
    )

    common = (
        des_codes & upag_codes
    )

    print(
        f"\n{crop}"
    )

    print(
        "DES Total districts:",
        len(des_codes)
    )

    print(
        "UPAg districts:",
        len(upag_codes)
    )

    print(
        "Common:",
        len(common)
    )


Rice
DES Total districts: 363
UPAg districts: 782
Common: 363

Wheat
DES Total districts: 5
UPAg districts: 782
Common: 5

Maize
DES Total districts: 393
UPAg districts: 782
Common: 393

Urad
DES Total districts: 338
UPAg districts: 782
Common: 338


In [18]:
#Inspecting actual matched records
rice_des = des_2022[
    (des_2022["crop"] == "Rice")
    &
    (des_2022["season"] == "Total")
].copy()

rice_upag = upag_2022[
    upag_2022["crop"] == "Rice"
].copy()

rice_match = pd.merge(
    rice_des,
    rice_upag,
    on=[
        "state_code",
        "district_code"
    ],
    how="inner",
    suffixes=("_des", "_upag")
)

print(
    rice_match[
        [
            "state_des",
            "district_des",
            "area_ha_des",
            "area_ha_upag",
            "production_tonnes_des",
            "production_tonnes_upag",
            "yield_kg_ha_des",
            "yield_kg_ha_upag"
        ]
    ].head(20)
)

         state_des                 district_des  area_ha_des  area_ha_upag  production_tonnes_des  production_tonnes_upag  yield_kg_ha_des  \
0   Andhra Pradesh        Alluri Sitharama Raju      58705.0       59000.0               175374.0                175000.0           2987.0   
1   Andhra Pradesh                   Anakapalli      59406.0       59000.0               186481.0                186000.0           3139.0   
2   Andhra Pradesh                Ananthapuramu      26995.0       27000.0               103988.0                104000.0           3852.0   
3   Andhra Pradesh                    Annamayya      15906.0       16000.0                52418.0                 52000.0           3295.0   
4   Andhra Pradesh                      Bapatla     115379.0      115000.0               480939.0                481000.0           4168.0   
5   Andhra Pradesh                     Chittoor      19888.0       20000.0                62697.0                 63000.0           3153.0   
6   An

In [19]:
# Identify UPAg-only states

upag_only = (
    upag_2022[
        upag_2022["state_code"].isin(
            sorted(upag_only_states)
        )
    ][
        [
            "state_code",
            "state"
        ]
    ]
    .drop_duplicates()
    .sort_values("state_code")
)

display(upag_only)

,state_code,state
769,7,Delhi
78,31,Lakshadweep


In [20]:
# Detailed DES season distribution for the four validation crops

validation_crops = [
    "Rice",
    "Wheat",
    "Maize",
    "Urad"
]

season_summary = (
    des_2022[
        des_2022["crop"].isin(validation_crops)
    ]
    .groupby(
        ["crop", "season"]
    )
    .agg(
        district_count=(
            "district_code",
            "nunique"
        ),
        observation_count=(
            "district_code",
            "size"
        ),
        total_area_ha=(
            "area_ha",
            "sum"
        ),
        total_production_tonnes=(
            "production_tonnes",
            "sum"
        )
    )
    .reset_index()
    .sort_values(
        ["crop", "district_count"],
        ascending=[True, False]
    )
)

display(season_summary)

,crop,season,district_count,observation_count,total_area_ha,total_production_tonnes
1,Maize,Kharif,552,552,7.491378e+06,2.241874e+07
4,Maize,Total,393,393,7.304143e+06,2.852768e+07
2,Maize,Rabi,351,351,2.191371e+06,1.232166e+07
3,Maize,Summer,214,214,5.062171e+05,2.641566e+06
0,Maize,Autumn,74,74,3.144490e+05,8.780856e+05
6,Maize,Winter,10,10,2.540000e+02,6.951000e+02
5,Maize,Whole Year,5,5,8.820000e+01,2.586700e+02
8,Rice,Kharif,485,485,2.855873e+07,8.812963e+07
11,Rice,Total,363,363,3.248316e+07,9.834238e+07
10,Rice,Summer,246,246,3.119536e+06,1.083685e+07


In [21]:
#Identifying dominant season for each crop
dominant_seasons = (
    season_summary
    .sort_values(
        ["crop", "district_count"],
        ascending=[True, False]
    )
    .groupby("crop")
    .head(1)
)

display(dominant_seasons)

,crop,season,district_count,observation_count,total_area_ha,total_production_tonnes
1,Maize,Kharif,552,552,7491378.00,2.241874e+07
8,Rice,Kharif,485,485,28558729.69,8.812963e+07
14,Urad,Kharif,488,488,3024839.18,1.788590e+06
20,Wheat,Rabi,537,537,35179300.39,1.284740e+08


In [22]:
#checking whether the dominant season has sensible annual coverage
for crop in validation_crops:

    crop_data = season_summary[
        season_summary["crop"] == crop
    ]

    print("\n" + "=" * 60)
    print(crop)
    print("=" * 60)

    display(crop_data)


Rice


,crop,season,district_count,observation_count,total_area_ha,total_production_tonnes
8,Rice,Kharif,485,485,2.855873e+07,8.812963e+07
11,Rice,Total,363,363,3.248316e+07,9.834238e+07
10,Rice,Summer,246,246,3.119536e+06,1.083685e+07
12,Rice,Winter,193,193,1.311415e+07,3.796920e+07
7,Rice,Autumn,176,176,1.550800e+06,3.913987e+06
9,Rice,Rabi,115,115,4.137540e+06,1.410341e+07



Wheat


,crop,season,district_count,observation_count,total_area_ha,total_production_tonnes
20,Wheat,Rabi,537,537,35179300.39,1.284740e+08
23,Wheat,Whole Year,9,9,2370.00,5.880000e+03
21,Wheat,Summer,7,7,223.00,3.452000e+02
19,Wheat,Kharif,5,5,26.00,5.300000e+01
22,Wheat,Total,5,5,1460.00,3.022000e+03



Maize


,crop,season,district_count,observation_count,total_area_ha,total_production_tonnes
1,Maize,Kharif,552,552,7491378.00,2.241874e+07
4,Maize,Total,393,393,7304143.49,2.852768e+07
2,Maize,Rabi,351,351,2191371.11,1.232166e+07
3,Maize,Summer,214,214,506217.12,2.641566e+06
0,Maize,Autumn,74,74,314449.00,8.780856e+05
6,Maize,Winter,10,10,254.00,6.951000e+02
5,Maize,Whole Year,5,5,88.20,2.586700e+02



Urad


,crop,season,district_count,observation_count,total_area_ha,total_production_tonnes
14,Urad,Kharif,488,488,3024839.18,1.788590e+06
17,Urad,Total,338,338,2959712.09,2.117207e+06
15,Urad,Rabi,239,239,765945.40,7.165234e+05
16,Urad,Summer,161,161,143271.60,1.516973e+05
18,Urad,Winter,28,28,57113.00,2.204850e+04
13,Urad,Autumn,15,15,12921.00,3.838900e+03


In [23]:
rice_match["area_diff"] = (
    rice_match["area_ha_des"]
    - rice_match["area_ha_upag"]
)

rice_match["production_diff"] = (
    rice_match["production_tonnes_des"]
    - rice_match["production_tonnes_upag"]
)

rice_match["yield_diff"] = (
    rice_match["yield_kg_ha_des"]
    - rice_match["yield_kg_ha_upag"]
)

print("Area difference:")
print(rice_match["area_diff"].describe())

print("\nProduction difference:")
print(rice_match["production_diff"].describe())

print("\nYield difference:")
print(rice_match["yield_diff"].describe())

Area difference:
count    363.000000
mean      -7.829069
std      287.968378
min     -500.000000
25%     -268.000000
50%        3.000000
75%      227.000000
max      553.000000
Name: area_diff, dtype: float64

Production difference:
count    363.000000
mean     -12.726322
std      274.237899
min     -499.000000
25%     -225.000000
50%      -15.000000
75%      196.500000
max      493.000000
Name: production_diff, dtype: float64

Yield difference:
count     363.000000
mean       -7.245179
std       138.039201
min     -2630.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.000000
Name: yield_diff, dtype: float64
